In [3]:
#assignment 
#create a snake game using reinforcement learning
# Simple Snake Game with Q-Learning (text-based)
import random
import numpy as np

grid_size = 6
actions = ["UP", "DOWN", "LEFT", "RIGHT"]
alpha = 0.1
gamma = 0.9
epsilon = 0.2

q_table = {}

def reset_game():
    snake = [(grid_size // 2, grid_size // 2)]
    food = (random.randrange(grid_size), random.randrange(grid_size))
    while food in snake:
        food = (random.randrange(grid_size), random.randrange(grid_size))
    direction = "RIGHT"
    return snake, food, direction

def get_state(snake, food):
    head_x, head_y = snake[0]
    food_x, food_y = food
    return (head_x, head_y, food_x, food_y)

def get_q_values(state):
    if state not in q_table:
        q_table[state] = np.zeros(len(actions))
    return q_table[state]

def move(head, action):
    x, y = head
    if action == "UP":
        return (x, y - 1)
    if action == "DOWN":
        return (x, y + 1)
    if action == "LEFT":
        return (x - 1, y)
    return (x + 1, y)

def step(snake, food, action):
    new_head = move(snake[0], action)

    # collision with wall or self
    if (
        new_head[0] < 0 or new_head[0] >= grid_size or
        new_head[1] < 0 or new_head[1] >= grid_size or
        new_head in snake
    ):
        return snake, food, -10, True

    snake = [new_head] + snake

    # eat food
    if new_head == food:
        reward = 10
        food = (random.randrange(grid_size), random.randrange(grid_size))
        while food in snake:
            food = (random.randrange(grid_size), random.randrange(grid_size))
        done = False
    else:
        snake.pop()
        reward = -0.1
        done = False

    return snake, food, reward, done

# Training
episodes = 300

for episode in range(episodes):
    snake, food, direction = reset_game()
    total_reward = 0

    for _ in range(100):
        state = get_state(snake, food)
        q_values = get_q_values(state)

        if random.random() < epsilon:
            action_idx = random.randrange(len(actions))
        else:
            action_idx = int(np.argmax(q_values))

        action = actions[action_idx]
        next_snake, next_food, reward, done = step(snake, food, action)
        next_state = get_state(next_snake, next_food)
        next_q_values = get_q_values(next_state)

        old_value = q_values[action_idx]
        next_max = np.max(next_q_values)

        q_values[action_idx] = old_value + alpha * (reward + gamma * next_max - old_value)

        snake, food = next_snake, next_food
        total_reward += reward

        if reward == -10:
            break

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode + 1}: total reward = {total_reward:.2f}")

# Demo after training
snake, food, direction = reset_game()
print("\nDemo run:")
for _ in range(20):
    state = get_state(snake, food)
    action_idx = int(np.argmax(get_q_values(state)))
    action = actions[action_idx]
    snake, food, reward, done = step(snake, food, action)
    print(f"Action: {action:5} | Head: {snake[0]} | Food: {food} | Reward: {reward}")
    if reward == -10:
        print("Game over")
        break

Episode 50: total reward = -11.20
Episode 100: total reward = -11.40
Episode 150: total reward = -10.90
Episode 200: total reward = -10.80
Episode 250: total reward = -10.40
Episode 300: total reward = -0.40

Demo run:
Action: DOWN  | Head: (3, 4) | Food: (5, 5) | Reward: -0.1
Action: LEFT  | Head: (2, 4) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 2) | Food: (5, 5) | Reward: -0.1
Action: DOWN  | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 2) | Food: (5, 5) | Reward: -0.1
Action: DOWN  | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 2) | Food: (5, 5) | Reward: -0.1
Action: DOWN  | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 2) | Food: (5, 5) | Reward: -0.1
Action: DOWN  | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    | Head: (2, 2) | Food: (5, 5) | Reward: -0.1
Action: DOWN  | Head: (2, 3) | Food: (5, 5) | Reward: -0.1
Action: UP    